# 05 — Error Analysis (Phase 3)

Where the **Tier 1 transformer** beats the shipped **TF-IDF + Logistic Regression
baseline** on the product task — and where it doesn't.

Background: the first (naive) transformer run *lost* the product task
(macro-F1 0.6832 vs baseline 0.7065). After fixing the imbalance handling and the
validation set, the Tier 1 run *wins* it (0.7386). This notebook re-scores the
saved Tier 1 adapter class by class to see **which classes that ~3-point win
comes from**, reads real complaints the two models disagree on, and checks whether
256-token truncation is costing the transformer on long narratives.

Runs on the Mac (CPU/MPS) or Colab. It only does a forward pass through the saved
model — no training. Plan:

1. Load the data.
2. Rebuild the baseline exactly as notebook 03.
3. Load the Tier 1 transformer adapter.
4. Score both on the natural test set.
5. Per-class F1, disagreement buckets, example complaints.
6. Length-vs-truncation slice + a light company-name memorization probe.
7. Save `reports/error_analysis_report.md`.

Monetary relief is a tie (see `reports/transformer_report.md`); this notebook
focuses on the product task, where the interesting disagreement is.

In [1]:
# 1) Setup. Work from the project root and load the same data the other
#    notebooks use. The test set is the natural (unbalanced) 34,694 rows.
import os
import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

train = pd.read_parquet("data/processed/train.parquet")   # balanced - to fit the baseline
test = pd.read_parquet("data/processed/test.parquet")      # natural mix - to score on
print("train:", len(train), " test:", len(test))
print("columns:", list(test.columns))

train: 47121  test: 34694
columns: ['complaint_id', 'date_received', 'product', 'sub_product', 'issue', 'sub_issue', 'company', 'state', 'consumer_complaint_narrative', 'is_monetary_relief', 'is_untimely', 'is_escalated']


In [2]:
# 2) Rebuild the shipped baseline exactly as notebook 03 did:
#    word + char TF-IDF -> Logistic Regression (class-weight balanced).
#    We refit here (a few seconds) instead of loading a saved model, so this
#    notebook stands on its own. Fit on train only, then score on test.
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression

train_text = train["consumer_complaint_narrative"]
test_text = test["consumer_complaint_narrative"]

word_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3,
                             max_features=60000, sublinear_tf=True)
X_train_word = word_tfidf.fit_transform(train_text)
X_test_word = word_tfidf.transform(test_text)

char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                             max_features=60000, sublinear_tf=True)
X_train_char = char_tfidf.fit_transform(train_text)
X_test_char = char_tfidf.transform(test_text)

X_train_both = hstack([X_train_word, X_train_char]).tocsr()
X_test_both = hstack([X_test_word, X_test_char]).tocsr()

baseline_product = LogisticRegression(max_iter=3000, class_weight="balanced",
                                      random_state=42)
baseline_product.fit(X_train_both, train["product"])
baseline_pred = baseline_product.predict(X_test_both)
print("baseline built and scored.")

baseline built and scored.


In [3]:
# 3) Sanity check: this should reproduce the baseline's product score from
#    notebook 03 / baseline_report.md (macro-F1 about 0.7065). If it does, we
#    know the baseline half of the comparison is faithful.
from sklearn.metrics import f1_score, accuracy_score

baseline_macro = f1_score(test["product"], baseline_pred, average="macro")
baseline_acc = accuracy_score(test["product"], baseline_pred)
print("baseline product  macro-F1:", round(baseline_macro, 4),
      "  accuracy:", round(baseline_acc, 4))

baseline product  macro-F1: 0.7065   accuracy: 0.8337


In [4]:
# 4) Load the Tier 1 transformer (saved LoRA adapter) for the product task.
#    Needs torch + transformers + peft. Uses GPU if present, else MPS, else CPU.
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

MODEL_NAME = "distilbert-base-uncased"
ADAPTER_DIR = "models/product_distilbert_lora"

# The label <-> number order MUST match how the adapter was trained in nb04:
#   product_names = sorted(train["product"].unique())
product_names = sorted(train["product"].unique().tolist())
label2id = {name: i for i, name in enumerate(product_names)}
id2label = {i: name for name, i in label2id.items()}
print(len(product_names), "product classes:", product_names)

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,
                                                          num_labels=len(product_names))
transformer_product = PeftModel.from_pretrained(base, ADAPTER_DIR).to(device)
transformer_product.eval()
print("adapter loaded from", ADAPTER_DIR)

9 product classes: ['Checking or savings account', 'Credit card or prepaid card', 'Credit reporting, credit repair services, or other personal consumer reports', 'Debt collection', 'Money transfer, virtual currency, or money service', 'Mortgage', 'Payday loan, title loan, or personal loan', 'Student loan', 'Vehicle loan or lease']
device: mps


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


adapter loaded from models/product_distilbert_lora


In [5]:
# 5) Predict the product class for every test complaint with the transformer.
#    We go in batches because 34k narratives will not fit through the model at
#    once. This is inference only - just a forward pass, same as scoring in nb04.
MAX_LENGTH = 256   # the same truncation the model was trained with
BATCH_SIZE = 64

def predict_product(texts):
    preds = []
    confs = []
    texts = list(texts)
    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start + BATCH_SIZE]
        enc = tokenizer(batch, truncation=True, padding="max_length",
                        max_length=MAX_LENGTH, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = transformer_product(**enc).logits
        probs = torch.softmax(logits, dim=1)
        conf, idx = probs.max(dim=1)
        preds.extend(idx.cpu().tolist())
        confs.extend(conf.cpu().tolist())
        if start % (BATCH_SIZE * 50) == 0:
            print("  scored", start, "/", len(texts))
    return preds, confs

pred_idx, pred_conf = predict_product(test["consumer_complaint_narrative"])
transformer_pred = [id2label[i] for i in pred_idx]
print("done - scored", len(transformer_pred), "complaints.")

  scored 0 / 34694
  scored 3200 / 34694
  scored 6400 / 34694
  scored 9600 / 34694
  scored 12800 / 34694
  scored 16000 / 34694
  scored 19200 / 34694
  scored 22400 / 34694
  scored 25600 / 34694
  scored 28800 / 34694
  scored 32000 / 34694
done - scored 34694 complaints.


In [6]:
# 6) The transformer's product score. Should land near the Tier 1 number in
#    transformer_report.md (macro-F1 about 0.7386).
transformer_macro = f1_score(test["product"], transformer_pred, average="macro")
transformer_acc = accuracy_score(test["product"], transformer_pred)
print("transformer product  macro-F1:", round(transformer_macro, 4),
      "  accuracy:", round(transformer_acc, 4))
print("baseline    product  macro-F1:", round(baseline_macro, 4))
print("difference (transformer - baseline):",
      round(transformer_macro - baseline_macro, 4))

transformer product  macro-F1: 0.7386   accuracy: 0.8537
baseline    product  macro-F1: 0.7065
difference (transformer - baseline): 0.0321


In [7]:
# 7) Per-class F1, baseline vs transformer, side by side. THIS is the piece
#    that was missing: which classes the transformer's ~3-point win comes from.
#    'support' is how many test complaints are in that class.
rows = []
for name in product_names:
    support = int((test["product"] == name).sum())
    b = f1_score(test["product"] == name, np.array(baseline_pred) == name,
                 zero_division=0)
    t = f1_score(test["product"] == name, np.array(transformer_pred) == name,
                 zero_division=0)
    winner = "transformer" if t > b else ("baseline" if b > t else "tie")
    rows.append([name, support, round(b, 3), round(t, 3), winner])

per_class = pd.DataFrame(rows, columns=["product", "support",
                                        "baseline_f1", "transformer_f1", "winner"])
per_class = per_class.sort_values("support", ascending=False).reset_index(drop=True)
print(per_class.to_string(index=False))
print()
print("classes won by transformer:",
      int((per_class["winner"] == "transformer").sum()), "of", len(per_class))

                                                                     product  support  baseline_f1  transformer_f1      winner
Credit reporting, credit repair services, or other personal consumer reports    22418        0.909           0.926 transformer
                                                             Debt collection     3427        0.704           0.715 transformer
                                                 Credit card or prepaid card     2863        0.737           0.734    baseline
                                                 Checking or savings account     2590        0.789           0.801 transformer
                                                                    Mortgage     1139        0.886           0.860    baseline
                          Money transfer, virtual currency, or money service      836        0.648           0.652 transformer
                                                       Vehicle loan or lease      678        0.586           0.

In [8]:
# 8) Where do the two models disagree? Bucket every test complaint.
truth = test["product"].tolist()
b_pred = list(baseline_pred)
t_pred = list(transformer_pred)

both_right = both_wrong = only_t = only_b = 0
for y, b, t in zip(truth, b_pred, t_pred):
    b_ok = (b == y)
    t_ok = (t == y)
    if b_ok and t_ok:
        both_right += 1
    elif not b_ok and not t_ok:
        both_wrong += 1
    elif t_ok and not b_ok:
        only_t += 1
    else:
        only_b += 1

print("both right:            ", both_right)
print("both wrong:            ", both_wrong)
print("transformer only right:", only_t)
print("baseline only right:   ", only_b)
print("net extra correct for the transformer:", only_t - only_b)

both right:             27668
both wrong:             3819
transformer only right: 1950
baseline only right:    1257
net extra correct for the transformer: 693


In [9]:
# 9) Read a few real complaints the transformer got right and the baseline got
#    wrong (and vice versa). Reading these is where the intuition comes from.
def show_examples(want_t_ok, want_b_ok, n=3):
    shown = 0
    for i in range(len(truth)):
        b_ok = (b_pred[i] == truth[i])
        t_ok = (t_pred[i] == truth[i])
        if t_ok == want_t_ok and b_ok == want_b_ok:
            text = str(test["consumer_complaint_narrative"].iloc[i]).replace("\n", " ")
            print("TRUE:", truth[i], "| baseline:", b_pred[i], "| transformer:", t_pred[i])
            print(text[:400], "...")
            print("-" * 80)
            shown += 1
            if shown >= n:
                break

print("### Transformer right, baseline wrong ###")
show_examples(True, False, n=3)
print()
print("### Baseline right, transformer wrong ###")
show_examples(False, True, n=3)

### Transformer right, baseline wrong ###
TRUE: Credit card or prepaid card | baseline: Checking or savings account | transformer: Credit card or prepaid card
I made a transfer of available funds from the card to my personal bank account and after 6 days my funds still hasn't showed up. There is no one to contact for this card no live rep as stated in their paperwork. There is only a automated system. I can't reach a live person to question what happened to my funds. ...
--------------------------------------------------------------------------------
TRUE: Checking or savings account | baseline: Money transfer, virtual currency, or money service | transformer: Checking or savings account
I have an account at Wells Fargo and attempted an EFT transfer to Truist of {$100.00} that was denied on XX/XX/19 even though I verified the transaction through text message. Now as of Last night Wells Fargo shut off my virtual card through G pay for no apparent reason and would not give me an explanat

In [10]:
# 10) Does 256-token truncation hurt the transformer on long narratives?
#     Measure each complaint's length in tokens, split into 'fits in 256' vs
#     'longer than 256', and compare each model's accuracy in each bucket.
#     Counting tokens for 34k rows is slow, so we sample for this slice only.
LEN_SAMPLE = 6000
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(test), size=min(LEN_SAMPLE, len(test)), replace=False)

lengths = []
for i in sample_idx:
    text = test["consumer_complaint_narrative"].iloc[i]
    lengths.append(len(tokenizer.encode(text, truncation=False, add_special_tokens=True)))
lengths = np.array(lengths)

b_ok_sample = np.array([b_pred[i] == truth[i] for i in sample_idx])
t_ok_sample = np.array([t_pred[i] == truth[i] for i in sample_idx])

def bucket(mask):
    return b_ok_sample[mask].mean(), t_ok_sample[mask].mean(), int(mask.sum())

short_mask = lengths <= 256
long_mask = lengths > 256
b_s, t_s, n_s = bucket(short_mask)
b_l, t_l, n_l = bucket(long_mask)
print("narratives that FIT in 256 tokens (n=%d):" % n_s)
print("   baseline acc %.3f   transformer acc %.3f" % (b_s, t_s))
print("narratives LONGER than 256 tokens (n=%d):" % n_l)
print("   baseline acc %.3f   transformer acc %.3f" % (b_l, t_l))
print()
print("If the transformer's edge shrinks on the long bucket, 256-token")
print("truncation is costing it there - it only sees the first 256 tokens,")
print("while TF-IDF reads the whole narrative.")

Token indices sequence length is longer than the specified maximum sequence length for this model (906 > 512). Running this sequence through the model will result in indexing errors


narratives that FIT in 256 tokens (n=3750):
   baseline acc 0.807   transformer acc 0.831
narratives LONGER than 256 tokens (n=2250):
   baseline acc 0.880   transformer acc 0.892

If the transformer's edge shrinks on the long bucket, 256-token
truncation is costing it there - it only sees the first 256 tokens,
while TF-IDF reads the whole narrative.


In [11]:
# 11) Light memorization / leakage probe (optional - only if a company column
#     exists). Narratives are mostly XXXX-masked, but when a complaint names its
#     own company in plain text, does the model lean on that? Compare accuracy on
#     rows where the company name appears unmasked in the text vs the rest.
if "company" in test.columns:
    def name_in_text(row):
        company = str(row["company"]) if pd.notna(row["company"]) else ""
        first_word = company.lower().split()[0] if company else ""
        narrative = str(row["consumer_complaint_narrative"]).lower()
        return len(first_word) >= 4 and first_word in narrative

    leaks = test.apply(name_in_text, axis=1).to_numpy()
    b_ok_all = np.array([b_pred[i] == truth[i] for i in range(len(truth))])
    t_ok_all = np.array([t_pred[i] == truth[i] for i in range(len(truth))])
    print("rows where the company name appears in the text:",
          int(leaks.sum()), "of", len(test))
    if leaks.sum() > 0:
        print("  name present -> baseline acc %.3f  transformer acc %.3f"
              % (b_ok_all[leaks].mean(), t_ok_all[leaks].mean()))
    print("  name absent  -> baseline acc %.3f  transformer acc %.3f"
          % (b_ok_all[~leaks].mean(), t_ok_all[~leaks].mean()))
    print("A big accuracy gap when the name is present hints the model uses the")
    print("company name as a shortcut rather than the complaint content.")
else:
    print("no 'company' column in test.parquet - skipping the memorization probe.")
    print("columns available:", list(test.columns))

rows where the company name appears in the text: 10878 of 34694
  name present -> baseline acc 0.822  transformer acc 0.841
  name absent  -> baseline acc 0.839  transformer acc 0.859
A big accuracy gap when the name is present hints the model uses the
company name as a shortcut rather than the complaint content.


In [12]:
# 12) Save a short report with the real numbers, same pattern as the other
#     notebooks. This is the human-readable Phase 3 error-analysis output.
lines = []
lines.append("# Phase 3 - Error Analysis (baseline vs Tier 1 transformer)")
lines.append("")
lines.append("Generated by notebooks/05_error_analysis.ipynb.")
lines.append("Scored on the natural test set (" + str(len(test)) + " rows).")
lines.append("")
lines.append("## Product macro-F1")
lines.append("")
lines.append("- Baseline (TF-IDF + LogReg word+char): " + str(round(baseline_macro, 4)))
lines.append("- Transformer (Tier 1, DistilBERT + LoRA): " + str(round(transformer_macro, 4)))
lines.append("- Difference (transformer - baseline): "
             + str(round(transformer_macro - baseline_macro, 4)))
lines.append("")
lines.append("## Per-class F1 (sorted by support)")
lines.append("")
lines.append("```")
lines.append(per_class.to_string(index=False))
lines.append("```")
lines.append("")
lines.append("## Where they disagree (product)")
lines.append("")
lines.append("- both right: " + str(both_right))
lines.append("- both wrong: " + str(both_wrong))
lines.append("- transformer only right: " + str(only_t))
lines.append("- baseline only right: " + str(only_b))
lines.append("- net extra correct for the transformer: " + str(only_t - only_b))
lines.append("")
lines.append("## Length vs 256-token truncation (sampled, n=" + str(len(sample_idx)) + ")")
lines.append("")
lines.append("- fits in 256 tokens (n=" + str(n_s) + "): baseline acc "
             + str(round(b_s, 3)) + ", transformer acc " + str(round(t_s, 3)))
lines.append("- longer than 256 tokens (n=" + str(n_l) + "): baseline acc "
             + str(round(b_l, 3)) + ", transformer acc " + str(round(t_l, 3)))
lines.append("")

report_path = "reports/error_analysis_report.md"
with open(report_path, "w") as f:
    f.write("\n".join(lines))
print("wrote", report_path)

wrote reports/error_analysis_report.md
